In [39]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

def indexes(my_list, desired_element):
    return [i+1 for i, e in enumerate(my_list) if e == desired_element]

def input_id_pos(input_word):
    # 1) Identity (35-dim)
    t = Tokenizer()
    fit_text = "ء أ ؤ إ ئ ا ب ت ة ث ج ح خ د ذ ر ز س ش ص ض ط ظ ع غ ف ق ك ل م ن ه و ى ي"
    t.fit_on_texts(fit_text)
    seqs = t.texts_to_sequences(input_word)
    flat = [val for sub in seqs for val in sub]
    identity = ['1' if idx in flat else '0'
                for idx in range(1, len(t.word_index) + 1)]

    # 2) Position (100-dim: 10 bits per letter)
    uniq = sorted(set(input_word))
    pos_flat = []
    for ch in uniq:
        for p in indexes(list(input_word), ch):
            pos_flat.append(p)
    position = []
    for p in pos_flat:
        val = 1 << (10 - p)
        bits = f'{val:010b}'
        position.extend(list(bits))
    if len(position) < 100:
        position.extend(['0'] * (100 - len(position)))

    return identity, position

def save_encodings(word_list, filename):
    # build DataFrames
    df_word = pd.DataFrame({'word': word_list})
    id_rows, pos_rows = [], []
    for w in word_list:
        ident, pos = input_id_pos(w)
        id_rows.append(ident)
        pos_rows.append(pos)
    df_id  = pd.DataFrame(id_rows)
    df_pos = pd.DataFrame(pos_rows)

    # set column names 1…35 and 1…100
    df_id.columns  = range(1, df_id.shape[1] + 1)
    df_pos.columns = range(1, df_pos.shape[1] + 1)

    # write three sheets: word, identity, position
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_word.to_excel(writer, sheet_name='word',      index=True)
        df_id.to_excel( writer, sheet_name='identity',  index=True)
        df_pos.to_excel(writer, sheet_name='position',  index=True)

In [40]:
transposed_words= pd.read_excel("F:/modelling_allographic_letter_recognition/sim3&4/Target3&4.xlsx")['TL'].to_list()
substituted_words= pd.read_excel("F:/modelling_allographic_letter_recognition/sim3&4/Target3&4.xlsx", sheet_name="SL")['SL'].to_list()

In [41]:
save_encodings(transposed_words,  'F:/modelling_allographic_letter_recognition/sim3&4/Transposed_idpos.xlsx')
save_encodings(substituted_words, 'F:/modelling_allographic_letter_recognition/sim3&4/Substituted_idpos.xlsx')

## Onehot Encoding

In [6]:
import pandas as pd
import numpy as np
from tensorflow.keras.utils import to_categorical

# Alphabet and one-hot lookup 
letters = "ءأؤإئابتةثجحخدذرزسشصضطظعغفقكلمنهوىي"
y_train_label = list(range(len(letters)))
y_train_label.reverse()
y_train_label_onehot = to_categorical(y_train_label)  # shape (35,35)

def onehotencod(word):
    encoded = []
    for ch in word:
        pos = letters.find(ch)
        if pos < 0:
            raise ValueError(f"Character {ch!r} not in letters alphabet")
        encoded.append(y_train_label_onehot[pos].tolist())
    flat = [val for sub in encoded for val in sub]
    if len(flat) < 350:
        flat.extend([0] * (350 - len(flat)))
    return flat

if __name__ == '__main__':

   
    word_list = ['الانفصالية', 'الإقليمية', 'البحرينية', 'التعليمات', 'التفسيرية', 'ينفذها', 'ينقذون', 'ينكرها', 'يسعدك', 'يشعرك']
    

    rows = [onehotencod(w) for w in word_list]

    # Sheet 1: vectors (v0–v349)
    df_vectors = pd.DataFrame(rows, columns=[f"v{i}" for i in range(350)])
    df_vectors.insert(0, "word", word_list)

    # Sheet 2: word + full vector as a single cell (like the author)
    df_onehot = pd.DataFrame({
        'onehotenc': [str(r) for r in rows],
        'Target':    word_list
    })

    output_file = 'E:/simulation-project/tests/test_onehot_encoding.xlsx'
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_vectors.to_excel(writer, sheet_name='vectors', index=True)
        df_onehot.to_excel( writer, sheet_name='onehot',  index=True)

    print(f"Saved to {output_file}")

Saved to E:/simulation-project/tests/test_onehot_encoding.xlsx
